In [1]:
import pandas as pd
import numpy as np

In [2]:
train_df = pd.read_csv("/kaggle/input/competitions/ai-generating-text-detection/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/ai-generating-text-detection/test.csv")

In [3]:
train_df.head()

,id,text,label
0,0,"[""Generally , it means they 've changed their ...",0
1,1,['It works by crowd sourcing WiFi addresses fr...,0
2,2,Would you like technology called the Facial Ac...,0
3,3,"[""Sure thing! The golden ratio is a special nu...",1
4,4,['The black-and-white series originally ran fr...,0


In [4]:
train_df.describe()

,id,label
count,73373.000000,73373.000000
mean,36686.000000,0.472531
std,21181.104988,0.499248
min,0.000000,0.000000
25%,18343.000000,0.000000
50%,36686.000000,0.000000
75%,55029.000000,1.000000
max,73372.000000,1.000000


In [6]:
train_df['label'].value_counts()

label
0    38702
1    34671
Name: count, dtype: int64

In [7]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [8]:
stop = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [19]:
def process_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\@\w+|\#', '', text)
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop and len(word) > 1]
    return ''.join(tokens)

In [20]:
train_df["processed_text"] = train_df["text"].apply(process_text)
test_df["processed_text"] = test_df["text"].apply(process_text)

In [21]:
print(train_df['text'].iloc[0])
print(train_df['processed_text'].iloc[0])

["Generally , it means they 've changed their focus from making music to making money .", 'It is usually used to refer to a band that makes rock or punk type music ( or any other type of music that is not pop ) . They are called " sellouts " when they start to make music that sounds more pop , making them more likable to the majority of the population and more likely to get in the top 40 on the radio for example . This leaves their original fans who supported them angry because they feel like the band only got well known because of their support but now the band does n\'t care about what they want to hear more of , only what will make the most money .', 'quick question , one of the most famous bands , green day , are sometimes referred to as sellouts . What kind are they considered ? For an example .']
generallymeanchangedfocusmakingmusicmakingmoneyusuallyusedreferbandmakerockpunktypemusictypemusicpopcalledselloutstartmakemusicsoundpopmakinglikablemajoritypopulationlikelygettopradioexa

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV

In [25]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')), 
    ('nb', MultinomialNB())
])

In [26]:
params_grid = {
    'tfidf__max_features': [5000, 10000, 15000, 20000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [1, 5, 8, 10],
    'tfidf__max_df': [0.7, 0.9],
    'nb__alpha': [0.1, 0.5, 1]
}

In [28]:
grid = GridSearchCV(pipeline, params_grid, cv=5, scoring='roc_auc', verbose=1, n_jobs=-1)

In [29]:
X_pipeline_train, X_val, y_pipeline_train, y_val = train_test_split(
    train_df['processed_text'], train_df['label'], test_size=0.2, random_state=42, stratify=train_df['label']
)

In [30]:
grid.fit(X_pipeline_train, y_pipeline_train)

Fitting 5 folds for each of 192 candidates, totalling 960 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
432 fits failed out of a total of 960.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
432 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 654, in fit
    Xt = self._fit(X, y, routed_params

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tfidf',
                                        TfidfVectorizer(stop_words='english')),
                                       ('nb', MultinomialNB())]),
             n_jobs=-1,
             param_grid={'nb__alpha': [0.1, 0.5, 1],
                         'tfidf__max_df': [0.7, 0.9],
                         'tfidf__max_features': [5000, 10000, 15000, 20000],
                         'tfidf__min_df': [1, 5, 8, 10],
                         'tfidf__ngram_range': [(1, 1), (1, 2)]},
             scoring='roc_auc', verbose=1)

In [32]:
print(f"Meilleurs hyperparamètres (MultinomialNB): {grid.best_params_}")
print(f"Meilleur score ROC AUC (CV - MultinomialNB): {grid.best_score_:.4f}")

Meilleurs hyperparamètres (MultinomialNB): {'nb__alpha': 0.1, 'tfidf__max_df': 0.7, 'tfidf__max_features': 20000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1)}
Meilleur score ROC AUC (CV - MultinomialNB): 0.5257


In [34]:
from sklearn.linear_model import LogisticRegression
import time
pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('lr', LogisticRegression(solver='liblinear', random_state=42)) 
])

param_grid_lr = {
    'tfidf__max_features': [5000, 10000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [1, 3],
    'tfidf__max_df': [0.7, 0.9],
    'lr__C': [0.1, 1.0, 10.0], 
    'lr__penalty': ['l1', 'l2'] 
}

grid_search_lr = GridSearchCV(
    pipeline_lr,
    param_grid_lr,
    cv=3,
    scoring='roc_auc',
    verbose=1,
    n_jobs=-1
)

start_time = time.time()
grid_search_lr.fit(X_pipeline_train, y_pipeline_train)
end_time = time.time()

print(f"Temps d'entraînement GridSearchCV (Logistic Regression): {end_time - start_time:.2f} secondes")
print(f"Meilleurs hyperparamètres (Logistic Regression): {grid_search_lr.best_params_}")
print(f"Meilleur score ROC AUC (CV - Logistic Regression): {grid_search_lr.best_score_:.4f}")

Fitting 3 folds for each of 96 candidates, totalling 288 fits
Temps d'entraînement GridSearchCV (Logistic Regression): 136.07 secondes
Meilleurs hyperparamètres (Logistic Regression): {'lr__C': 0.1, 'lr__penalty': 'l2', 'tfidf__max_df': 0.7, 'tfidf__max_features': 10000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1)}
Meilleur score ROC AUC (CV - Logistic Regression): 0.5129


In [35]:
from sklearn.svm import LinearSVC
pipeline_svc = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('svc', LinearSVC(random_state=42, dual=False))
])

param_grid_svc = {
    'tfidf__max_features': [5000, 10000, 20000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [1, 3],
    'tfidf__max_df': [0.7, 0.9],
    'svc__C': [0.1, 1.0, 5.0, 10.0]
}

grid_search_svc = GridSearchCV(
    pipeline_svc,
    param_grid_svc,
    cv=3,
    scoring='roc_auc',
    verbose=1,
    n_jobs=-1
)

start_time = time.time()
grid_search_svc.fit(X_pipeline_train, y_pipeline_train)
end_time = time.time()

print(f"Temps d'entraînement GridSearchCV (Linear SVC): {end_time - start_time:.2f} secondes")
print(f"Meilleurs hyperparamètres (Linear SVC): {grid_search_svc.best_params_}")
print(f"Meilleur score ROC AUC (CV - Linear SVC): {grid_search_svc.best_score_:.4f}")

Fitting 3 folds for each of 96 candidates, totalling 288 fits
Temps d'entraînement GridSearchCV (Linear SVC): 142.80 secondes
Meilleurs hyperparamètres (Linear SVC): {'svc__C': 0.1, 'tfidf__max_df': 0.7, 'tfidf__max_features': 20000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1)}
Meilleur score ROC AUC (CV - Linear SVC): 0.5258


In [36]:
best_nb = grid.best_estimator_
best_lr = grid_search_lr.best_estimator_
best_svc = grid_search_svc.best_estimator_

In [39]:
from sklearn.metrics import confusion_matrix, classification_report
models = {
    "MultinomialNB": best_nb,
    "LogisticRegression": best_lr,
    "LinearSVC": best_svc
}

best_roc_auc_overall = 0
final_best_model = None

for name, model in models.items():
    print(f"\nÉvaluation du modèle: {name}")
    if hasattr(model, 'predict_proba'):
        y_pred_proba = model.predict_proba(X_val)[:, 1]
    else: 
        y_pred_proba = model.decision_function(X_val)

    roc_auc = roc_auc_score(y_val, y_pred_proba)
    print(f"ROC AUC sur l'ensemble de validation ({name}): {roc_auc:.4f}")

    y_pred_binary = (y_pred_proba > 0.5).astype(int) if hasattr(model, 'predict_proba') else model.predict(X_val)

    print("\nMatrice de Confusion:")
    print(confusion_matrix(y_val, y_pred_binary, labels=[0, 1])) 

    print("\nRapport de Classification:")
    print(classification_report(y_val, y_pred_binary, target_names=['Human (0)', 'AI (1)']))

    if roc_auc > best_roc_auc_overall:
        best_roc_auc_overall = roc_auc
        final_best_model = model
        print(f"--> {name} est actuellement le meilleur modèle.")

print(f"\nLe modèle globalement choisi est: {final_best_model.__class__.__name__} avec un ROC AUC de {best_roc_auc_overall:.4f}")


Évaluation du modèle: MultinomialNB
ROC AUC sur l'ensemble de validation (MultinomialNB): 0.5276

Matrice de Confusion:
[[7741    0]
 [6817  117]]

Rapport de Classification:
              precision    recall  f1-score   support

   Human (0)       0.53      1.00      0.69      7741
      AI (1)       1.00      0.02      0.03      6934

    accuracy                           0.54     14675
   macro avg       0.77      0.51      0.36     14675
weighted avg       0.75      0.54      0.38     14675

--> MultinomialNB est actuellement le meilleur modèle.

Évaluation du modèle: LogisticRegression
ROC AUC sur l'ensemble de validation (LogisticRegression): 0.5152

Matrice de Confusion:
[[7741    0]
 [6923   11]]

Rapport de Classification:
              precision    recall  f1-score   support

   Human (0)       0.53      1.00      0.69      7741
      AI (1)       1.00      0.00      0.00      6934

    accuracy                           0.53     14675
   macro avg       0.76      0.50     

In [55]:
import pandas as pd
train = pd.read_csv("/kaggle/input/competitions/ai-generating-text-detection/train.csv")
test = pd.read_csv("/kaggle/input/competitions/ai-generating-text-detection/test.csv")
X_train = train['text'].values
y_train = train['label'].values
X_test  = test['text'].values

In [41]:
train["word_len"] = train["text"].str.len()
train["word_count"] = train["text"].str.split().str.len()

In [43]:
from collections import Counter

In [44]:
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 47.8 MB/s eta 0:00:0000:01


In [53]:
import textstat
def extract_features(text):
    words = text.split()
    sentences = [s.strip() for s in re.split(r'[.!?]', text) if s.strip()]

    ttr = len(set(words)) / len(words) if words else 0

    counts = Counter(words)
    hapax = sum(1 for w, c in counts.items() if c == 1) / len(words) if words else 0

    avg_sent_len = np.mean([len(s.split()) for s in sentences]) if sentences else 0
    std_sent_len = np.std([len(s.split()) for s in sentences]) if sentences else 0

    return {
        'text_len'        : len(text),
        'word_count'      : len(words),
        'avg_word_len'    : np.mean([len(w) for w in words]) if words else 0,
        'avg_sent_len'    : avg_sent_len,
        'std_sent_len'    : std_sent_len,
        'type_token_ratio': ttr,
        'hapax_ratio'     : hapax,
        'comma_rate'      : text.count(',')  / len(words) if words else 0,
        'semicolon_rate'  : text.count(';')  / len(words) if words else 0,
        'exclaim_rate'    : text.count('!')  / len(words) if words else 0,
        'flesch'          : textstat.flesch_reading_ease(text),
        'gunning_fog'     : textstat.gunning_fog(text),
    }

In [56]:
feat_train = pd.DataFrame([extract_features(t) for t in X_train])
feat_test  = pd.DataFrame([extract_features(t) for t in X_test])

In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack

tfidf_word = TfidfVectorizer(ngram_range=(1,3), max_features=100000,
                              sublinear_tf=True, analyzer='word')
tfidf_char = TfidfVectorizer(ngram_range=(2,4), max_features=50000,
                              sublinear_tf=True, analyzer='char_wb')

X_tfidf_train = hstack([tfidf_word.fit_transform(X_train),
                         tfidf_char.fit_transform(X_train)])
X_tfidf_test  = hstack([tfidf_word.transform(X_test),
                         tfidf_char.transform(X_test)])

lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_tfidf_train, y_train)

from sklearn.model_selection import cross_val_predict

tfidf_lr_probs_train = cross_val_predict(lr, X_tfidf_train, y_train,
                                          cv=5, method='predict_proba')[:, 1]
lr.fit(X_tfidf_train, y_train)
tfidf_lr_probs_test = lr.predict_proba(X_tfidf_test)[:, 1]

In [58]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_predict

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=42
)

xgb_probs_train = cross_val_predict(xgb, feat_train, y_train,
                                     cv=5, method='predict_proba')[:, 1]

xgb.fit(feat_train, y_train)
xgb_probs_test = xgb.predict_proba(feat_test)[:, 1]

In [59]:
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

device = 'cuda' if torch.cuda.is_available() else 'cpu'

ppl_model     = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
ppl_tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
ppl_model.eval()

def compute_perplexity(text):
    inputs = ppl_tokenizer(text, return_tensors='pt',
                            truncation=True, max_length=512).to(device)
    with torch.no_grad():
        loss = ppl_model(**inputs, labels=inputs['input_ids']).loss
    return torch.exp(loss).item()

perp_train = np.array([compute_perplexity(t) for t in X_train])
perp_test  = np.array([compute_perplexity(t) for t in X_test])

from sklearn.preprocessing import MinMaxScaler

perp_all = np.concatenate([perp_train, perp_test]).reshape(-1, 1)
scaler   = MinMaxScaler()
scaler.fit(perp_all)

perp_train_norm = 1 - scaler.transform(perp_train.reshape(-1,1)).flatten()
perp_test_norm  = 1 - scaler.transform(perp_test.reshape(-1,1)).flatten()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


In [63]:
import torch
import numpy as np
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           Trainer, TrainingArguments)
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedKFold
from scipy.special import softmax

tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')

class TextDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=512):
        self.encodings = tokenizer(list(texts), truncation=True,
                                    padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self): 
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(int(self.labels[idx]))
        return item

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
deberta_probs_train = np.zeros(len(X_train))
deberta_probs_test  = np.zeros(len(X_test))

# Détecte automatiquement bf16 ou pas
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"--- Fold {fold+1} ---")

    train_ds = TextDataset(X_train[tr_idx], y_train[tr_idx])
    val_ds   = TextDataset(X_train[val_idx], y_train[val_idx])
    test_ds  = TextDataset(X_test)

    model = AutoModelForSequenceClassification.from_pretrained(
        'microsoft/deberta-v3-base', num_labels=2)

    args = TrainingArguments(
        output_dir=f'./deberta_fold{fold}',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='no',
        learning_rate=2e-5,
        warmup_steps=100,        # ← remplace warmup_ratio (deprecated)
        fp16=False,              # ← DÉSACTIVÉ (incompatible DeBERTa-v3)
        bf16=use_bf16,           # ← activé seulement si GPU compatible
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds
    )
    trainer.train()

    val_preds = trainer.predict(val_ds).predictions
    deberta_probs_train[val_idx] = softmax(val_preds, axis=1)[:, 1]

    test_preds = trainer.predict(test_ds).predictions
    deberta_probs_test += softmax(test_preds, axis=1)[:, 1] / 5

--- Fold 1 ---


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias          

Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan


--- Fold 2 ---


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias          

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

train = pd.read_csv('/kaggle/input/competitions/ai-generating-text-detection/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/ai-generating-text-detection/test.csv')   

X_train = train['text'].values
y_train = train['label'].values
X_test  = test['text'].values

print(train['label'].value_counts())
print(train.head())

label
0    38702
1    34671
Name: count, dtype: int64
   id                                               text  label
0   0  ["Generally , it means they 've changed their ...      0
1   1  ['It works by crowd sourcing WiFi addresses fr...      0
2   2  Would you like technology called the Facial Ac...      0
3   3  ["Sure thing! The golden ratio is a special nu...      1
4   4  ['The black-and-white series originally ran fr...      0


In [3]:
!pip install textstat
import textstat
import re
from collections import Counter

def extract_features(text):
    words = text.split()
    sentences = [s.strip() for s in re.split(r'[.!?]', text) if s.strip()]

    ttr = len(set(words)) / len(words) if words else 0

    counts = Counter(words)
    hapax = sum(1 for w, c in counts.items() if c == 1) / len(words) if words else 0

    avg_sent_len = np.mean([len(s.split()) for s in sentences]) if sentences else 0
    std_sent_len = np.std([len(s.split()) for s in sentences]) if sentences else 0

    return {
        'text_len'        : len(text),
        'word_count'      : len(words),
        'avg_word_len'    : np.mean([len(w) for w in words]) if words else 0,
        'avg_sent_len'    : avg_sent_len,
        'std_sent_len'    : std_sent_len,
        'type_token_ratio': ttr,
        'hapax_ratio'     : hapax,
        'comma_rate'      : text.count(',')  / len(words) if words else 0,
        'semicolon_rate'  : text.count(';')  / len(words) if words else 0,
        'exclaim_rate'    : text.count('!')  / len(words) if words else 0,
        'flesch'          : textstat.flesch_reading_ease(text),
        'gunning_fog'     : textstat.gunning_fog(text),
    }

import pandas as pd
feat_train = pd.DataFrame([extract_features(t) for t in X_train])
feat_test  = pd.DataFrame([extract_features(t) for t in X_test])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 55.1 MB/s eta 0:00:0000:01


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack

tfidf_word = TfidfVectorizer(ngram_range=(1,3), max_features=100000,
                              sublinear_tf=True, analyzer='word')
tfidf_char = TfidfVectorizer(ngram_range=(2,4), max_features=50000,
                              sublinear_tf=True, analyzer='char_wb')

X_tfidf_train = hstack([tfidf_word.fit_transform(X_train),
                         tfidf_char.fit_transform(X_train)])
X_tfidf_test  = hstack([tfidf_word.transform(X_test),
                         tfidf_char.transform(X_test)])

lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_tfidf_train, y_train)

from sklearn.model_selection import cross_val_predict

tfidf_lr_probs_train = cross_val_predict(lr, X_tfidf_train, y_train,
                                          cv=5, method='predict_proba')[:, 1]
lr.fit(X_tfidf_train, y_train)
tfidf_lr_probs_test = lr.predict_proba(X_tfidf_test)[:, 1]

In [6]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_predict

xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.04,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=42
)

xgb_probs_train = cross_val_predict(xgb, feat_train, y_train,
                                     cv=5, method='predict_proba')[:, 1]

xgb.fit(feat_train, y_train)
xgb_probs_test = xgb.predict_proba(feat_test)[:, 1]

In [7]:
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

device = 'cuda' if torch.cuda.is_available() else 'cpu'

ppl_model     = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
ppl_tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
ppl_model.eval()

def compute_perplexity(text):
    inputs = ppl_tokenizer(text, return_tensors='pt',
                            truncation=True, max_length=512).to(device)
    with torch.no_grad():
        loss = ppl_model(**inputs, labels=inputs['input_ids']).loss
    return torch.exp(loss).item()

perp_train = np.array([compute_perplexity(t) for t in X_train])
perp_test  = np.array([compute_perplexity(t) for t in X_test])

from sklearn.preprocessing import MinMaxScaler

perp_all = np.concatenate([perp_train, perp_test]).reshape(-1, 1)
scaler   = MinMaxScaler()
scaler.fit(perp_all)

perp_train_norm = 1 - scaler.transform(perp_train.reshape(-1,1)).flatten()
perp_test_norm  = 1 - scaler.transform(perp_test.reshape(-1,1)).flatten()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


In [8]:
import torch
import numpy as np
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           Trainer, TrainingArguments)
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedKFold
from scipy.special import softmax

tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')

class TextDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=512):
        self.encodings = tokenizer(list(texts), truncation=True,
                                    padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self): 
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(int(self.labels[idx]))
        return item

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
deberta_probs_train = np.zeros(len(X_train))
deberta_probs_test  = np.zeros(len(X_test))

# Détecte automatiquement bf16 ou pas
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"--- Fold {fold+1} ---")

    train_ds = TextDataset(X_train[tr_idx], y_train[tr_idx])
    val_ds   = TextDataset(X_train[val_idx], y_train[val_idx])
    test_ds  = TextDataset(X_test)

    model = AutoModelForSequenceClassification.from_pretrained(
        'microsoft/deberta-v3-base', num_labels=2)

    args = TrainingArguments(
        output_dir=f'./deberta_fold{fold}',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='no',
        learning_rate=2e-5,
        warmup_steps=100,        # ← remplace warmup_ratio (deprecated)
        fp16=False,              # ← DÉSACTIVÉ (incompatible DeBERTa-v3)
        bf16=use_bf16,           # ← activé seulement si GPU compatible
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds
    )
    trainer.train()

    val_preds = trainer.predict(val_ds).predictions
    deberta_probs_train[val_idx] = softmax(val_preds, axis=1)[:, 1]

    test_preds = trainer.predict(test_ds).predictions
    deberta_probs_test += softmax(test_preds, axis=1)[:, 1] / 5

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

--- Fold 1 ---


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan


--- Fold 3 ---


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight      

Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan


In [10]:
print("deberta_probs_train NaN:", np.isnan(deberta_probs_train).sum())
print("xgb_probs_train     NaN:", np.isnan(xgb_probs_train).sum())
print("tfidf_lr_probs_train NaN:", np.isnan(tfidf_lr_probs_train).sum())
print("perp_train_norm     NaN:", np.isnan(perp_train_norm).sum())

print("\ndeberta_probs_test  NaN:", np.isnan(deberta_probs_test).sum())
print("xgb_probs_test      NaN:", np.isnan(xgb_probs_test).sum())
print("tfidf_lr_probs_test NaN:", np.isnan(tfidf_lr_probs_test).sum())
print("perp_test_norm      NaN:", np.isnan(perp_test_norm).sum())

deberta_probs_train NaN: 73373
xgb_probs_train     NaN: 0
tfidf_lr_probs_train NaN: 0
perp_train_norm     NaN: 0

deberta_probs_test  NaN: 19346
xgb_probs_test      NaN: 0
tfidf_lr_probs_test NaN: 0
perp_test_norm      NaN: 0


In [11]:
from scipy.special import softmax
meta_train = np.column_stack([
    xgb_probs_train,
    tfidf_lr_probs_train,
    perp_train_norm,
])

meta_test = np.column_stack([
    xgb_probs_test,
    tfidf_lr_probs_test,
    perp_test_norm,
])

meta = LogisticRegression(C=0.1)
meta.fit(meta_train, y_train)

print("AUC meta :", roc_auc_score(y_train, meta.predict_proba(meta_train)[:,1]))

final_probs = meta.predict_proba(meta_test)[:, 1]

AUC meta : 0.9984181648489623


In [15]:
submission = pd.DataFrame({
    'id'   : test['id'],
    'label': final_probs  
})
submission.to_csv('submission.csv', index=False)

In [16]:
from IPython.display import FileLink
FileLink('submission.csv')

/kaggle/working/submission.csv